In [13]:
import os
import time
import numpy as np
import scanpy as sc
import scvi
from scipy import sparse

adata_path = "/old_nfs/chengwang_data/ICML_data/real_data/t-cell-depleted-bm-rna.h5ad"
adata = sc.read_h5ad(adata_path)

adata_scvi = adata.copy()
if "counts" not in adata_scvi.layers:
    if sparse.issparse(adata_scvi.X):
        adata_scvi.layers["counts"] = adata_scvi.X.copy()
    else:
        adata_scvi.layers["counts"] = np.asarray(adata_scvi.X).copy()

scvi.settings.seed = 0

scvi.model.SCVI.setup_anndata(
    adata_scvi,
    layer="counts",
)

model = scvi.model.SCVI(adata_scvi)
model.view_anndata_setup()

start = time.time()
model.train()
end = time.time()

runtime_seconds = end - start
print(f"scVI runtime (seconds): {runtime_seconds:.2f}")
print(f"scVI runtime (minutes): {runtime_seconds/60:.2f}")

adata_scvi.obsm["X_scVI"] = model.get_latent_representation()
print(adata_scvi.obsm["X_scVI"].shape)

out_dir = "/home/chongxiao/projects/InfoGlobe/scvi_results"
os.makedirs(out_dir, exist_ok=True)
np.save(os.path.join(out_dir, "X_scVI.npy"), adata_scvi.obsm["X_scVI"])
adata_scvi.write(os.path.join(out_dir, "t_cell_depleted_bm_rna_scvi_output.h5ad"))
model.save(os.path.join(out_dir, "scvi_model"), overwrite=True)

Seed set to 0


Anndata setup with scvi-tools version 1.3.3.

Setup via `SCVI.setup_anndata` with arguments:

{
│   'layer': 'counts',
│   'batch_key': None,
│   'labels_key': None,
│   'size_factor_key': None,
│   'categorical_covariate_keys': None,
│   'continuous_covariate_keys': None
}

         Summary Statistics         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃     Summary Stat Key     ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│         n_batch          │   1   │
│         n_cells          │ 8627  │
│ n_extra_categorical_covs │   0   │
│ n_extra_continuous_covs  │   0   │
│         n_labels         │   1   │
│          n_vars          │ 17226 │
└──────────────────────────┴───────┘

...

                     labels State Registry                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃      Source Location      ┃ Categories ┃ scvi-tools Encoding ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ adata.obs['_scvi_labels'] │     0      │          0          │
└───────────────────────────┴────────────┴─────────────────────┘

Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
/home/chongxiao/miniconda3/envs/htodemux/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/chongxiao/miniconda3/envs/htodemux/lib/python3 ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/chongxiao/miniconda3/envs/htodemux/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` li

Training:   0%|          | 0/400 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=400` reached.


scVI runtime (seconds): 234.28
scVI runtime (minutes): 3.90
(8627, 10)
